In [ ]:
# Placeholder parameters: illustrative values only, not the ones used in live research.
LB, UB = 0.4, 0.6

# NEAT implementation for VIX, SPY

The blocks are structured as follows:

X   Install dependencies
X   The Black Scholes Model for calls and puts
X   GARCH implementation with variable rolling window
X   Main training code and NEAT implementation
X   Plot training results as cumulative PnL alongside SPY and VIX


In [ ]:
!pip install neat-python


# Black-Scholes Model

Inputs: Spot Price, Options Strike Price, Time to Expiry, Risk Free Interest Rate, Implied Volatility, option type

Outputs: Options contract price

In [ ]:
import numpy as np
from scipy.stats import norm

# S = asset spot price, K = options strike price, T = maturity in years
# r = risk free interest rate, sigma = annualized implied volatility, option_type = "call" or "put"
def black_scholes(S, K, T, r, sigma, option_type="call"):
    # Prevent division by zero
    if sigma <= 0 or T <= 0:
        if option_type == "call":
            return max(0, S - K * np.exp(-r * T))
        else:
            return max(0, K * np.exp(-r * T) - S)

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    return price

# Example usage
if __name__ == "__main__":
    S = 100    # stock price
    K = 100    # strike price
    T = 1      # X year to maturity
    r = 0.05   # X% risk-free rate
    sigma = 0.2  # X% annualized volatility

    call_price = black_scholes(S, K, T, r, sigma, "call")
    put_price = black_scholes(S, K, T, r, sigma, "put")

    print(f"Call Price: {call_price:.4f}")
    print(f"Put Price: {put_price:.4f}")


# GARCH implementation

Inputs: SPY csv file, rolling window length

Outputs: Time series data on the implied volatility on a certain day

This section is an example of how to use the python package arch to find the GARCH rolling window IV.

In [ ]:
import pandas as pd
import numpy as np
from arch import arch_model
import sys

def rolling_garch_iv(file_path, window=252):
    """
    Compute daily rolling GARCH implied volatility.

    Parameters
    ----------
    file_path : str
        Path to CSV file with historical data (must include 'Date' and 'Price' columns).
    window : int
        Rolling window size in days.

    Returns
    -------
    pd.DataFrame
        DataFrame with Date, Return, and Rolling GARCH IV (annualized).
    """

    # Load data
    df = pd.read_csv(file_path)

    # Try to standardize column names
    df.columns = [c.strip().lower() for c in df.columns]
    if 'date' not in df.columns:
        raise ValueError("CSV must contain a 'Date' column.")

    # Detect price column
    price_col = None
    for candidate in ['price', 'close', 'adj close']:
        if candidate in df.columns:
            price_col = candidate
            break
    if price_col is None:
        raise ValueError("CSV must contain a price column (Price, Close, or Adj Close).")

    # Ensure date is datetime and sort
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    # Compute log returns
    df['return'] = np.log(df[price_col]).diff() * 100  # percentage returns

    # Prepare container
    iv_series = []

    # Rolling GARCH fit
    for i in range(window, len(df)):
        window_data = df['return'].iloc[i-window+1:i+1].dropna()
        if len(window_data) < window:
            iv_series.append(np.nan)
            continue

        try:
            am = arch_model(window_data, vol='Garch', p=1, q=1, dist='normal')
            res = am.fit(disp="off")
            forecast = res.forecast(horizon=1)

            # Conditional variance forecast for next day
            variance = forecast.variance.values[-1, 0]
            daily_vol = np.sqrt(variance)

            # Annualize (assuming X trading days)
            annualized_vol = daily_vol * np.sqrt(252)

            iv_series.append(annualized_vol)
        except Exception as e:
            print(f"Error at index {i}: {e}", file=sys.stderr)
            iv_series.append(np.nan)

    # Align results
    df['garch_iv'] = [np.nan]*window + iv_series

    return df[['date', 'return', 'garch_iv']]

# Example usage
if __name__ == "__main__":
    file_path = "S&P 500 Historical Data 09122010 to 09112025.csv"
    result = rolling_garch_iv(file_path, window=252)
    result.to_csv("sp500_garch_iv.csv", index=False)
    print(result.tail(10))

Training

Inputs:
VIX csv file,
SPY csv file,
neat-config.txt

Outputs:
A forward looking neural network that classifies each day into long, short, neutral

This section is a basic implementation of NEAT to train a long-short strategy finder using VIX and SPY historical data with a X day rolling window to decide on whether to buy or sell SPY stock every day.

In [ ]:
import os
import neat
import numpy as np
import pandas as pd
import pickle
import multiprocessing
import random

# X. Load and preprocess data
spy_df = pd.read_csv(
    'SPY ETF Stock Price History 2021 to 2025.csv',
    parse_dates=['Date'],
    index_col='Date'
)
vix_df = pd.read_csv(
    'CBOE Volatility Index Historical Data 2021 to 2025.csv',
    parse_dates=['Date'],
    index_col='Date'
)
# Align SPY Close/Open/VIX Close
data = pd.concat([
    spy_df[['Open', 'Price']].rename(columns={'Price': 'SPY_Close', 'Open': 'SPY_Open'}),
    vix_df['Price'].rename('VIX_Close')
], axis=1).dropna()

# Build features & price_diff (Close - Open)
p = 10  # number of lags
df = data[['SPY_Close', 'VIX_Close']].pct_change().dropna()
dates = df.index[p:-1]

X_list, price_diff_list = [], []
for t in range(p, len(df) - 1):
    # feature window: last p days of SPY% and VIX% returns
    window = df[['SPY_Close', 'VIX_Close']].iloc[t - p:t].values.flatten()
    X_list.append(window.astype(np.float32))
    # next-day price change: SPY_Close_{t+X} - SPY_Open_{t+X}
    date_next = df.index[t + 1]
    diff = (data.at[date_next, 'SPY_Close'] - data.at[date_next, 'SPY_Open'])
    price_diff_list.append(np.float32(diff))

X = np.vstack(X_list)  # shape (N, X*p)
price_diff = np.array(price_diff_list)  # shape (N,)


def eval_genome(genome, config):
    net = neat.nn.FeedForwardNetwork.create(genome, config)
    # X. Raw outputs
    raw = np.array([net.activate(x)[0] for x in X], dtype=np.float32)
    # X. Daily PnL
    lb, ub = LB, UB
    signs = np.where(raw > ub, 1.0, np.where(raw < lb, -1.0, 0.0))
    pnl_steps = signs * price_diff
    # print(pnl_steps)
    # X. Annualized Sharpe (√X factor)
    mu = pnl_steps.mean()
    sigma = pnl_steps.std(ddof=1) + 1e-8
    sharpe = mu / sigma * np.sqrt(252)
    return float(sharpe)


def run_neat():
    # X. Load NEAT config
    config = neat.Config(
        neat.DefaultGenome,
        neat.DefaultReproduction,
        neat.DefaultSpeciesSet,
        neat.DefaultStagnation,
        'neat-config.txt'
    )

    pop = neat.Population(config)

    pop.add_reporter(neat.StdOutReporter(True))
    pop.add_reporter(neat.StatisticsReporter())
    pe = neat.ParallelEvaluator(10, eval_genome)
    winner = pop.run(pe.evaluate, n=100)  # X generations
    print('\nBest genome:\n', winner)

    # X. Save best genome
    with open('best_genome.pkl', 'wb') as f:
        pickle.dump(winner, f)


if __name__ == '__main__':
    multiprocessing.freeze_support()
    # optional: allow duplicate OpenMP libs
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
    run_neat()


Plot PnL with VIX and SPY

In [ ]:
import pandas as pd
import numpy as np
import neat
import pickle
import matplotlib.pyplot as plt

# X. Load data
vix_df = pd.read_csv('CBOE Volatility Index Historical Data 2006 to 2025.csv',
                     parse_dates=['Date'], index_col='Date')
spy_df = pd.read_csv('SPY ETF Stock Price History 2006 to 2025.csv',
                     parse_dates=['Date'], index_col='Date')
spy_df['Return'] = spy_df['Price'] - spy_df['Open']
data = pd.concat([spy_df['Price'].rename('SPY'),
                  vix_df['Price'].rename('VIX')], axis=1).dropna()

# X. Reconstruct X, returns, and simulate PnL with your best genome
p = 10
df = data.pct_change().dropna()
dates = df.index[p:-1]

X_list, price_diff_list = [], []
for t in range(p, len(df) - 1):
    # feature window: last p days of SPY% and VIX% returns
    window = df[['SPY', 'VIX']].iloc[t - p:t].values.flatten()
    X_list.append(window.astype(np.float32))
    # next-day price change: SPY_Close_{t+X} - SPY_Open_{t+X}
    date_next = df.index[t + 1]
    diff = spy_df.at[date_next, 'Return']
    price_diff_list.append(np.float32(diff))

X = np.vstack(X_list)  # shape (N, X*p)
price_diff = np.array(price_diff_list)  # shape (N,)

# Load best genome and build network
config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                     neat.DefaultSpeciesSet, neat.DefaultStagnation,
                     'neat-config.txt')
with open('best_genome.pkl', 'rb') as f:
    winner = pickle.load(f)
net = neat.nn.FeedForwardNetwork.create(winner, config)

# Simulate strategy
raw = np.array([net.activate(x)[0] for x in X], dtype=np.float32)
lb, ub = LB, UB
signs = np.where(raw > ub, 1.0, np.where(raw < lb, -1.0, 0.0))
# print(price_diff)
pnl_steps = signs * price_diff
cum_pnl = np.cumsum(pnl_steps)
print(np.mean(cum_pnl)/np.std(cum_pnl))

# # X. Plot side-by-side
# fig, axs = plt.subplots(X, X, figsize=(X, X))
#
# axs[X].plot(data.index, data['SPY'])
# axs[X].set_title('SPY Closing Price')
# axs[X].set_xlabel('Date')
# axs[X].set_ylabel('Price')
# axs[X].grid(True)
#
# axs[X].plot(dates, cum_pnl, color='orange')
# axs[X].set_title('Evolved Strategy Cumulative PnL')
# axs[X].set_xlabel('Date')
# axs[X].set_ylabel('Cumulative PnL')
# axs[X].grid(True)
#
# plt.tight_layout()
# plt.show()

fig, ax1 = plt.subplots(figsize=(10, 5))
# SPY price
ax1.plot(data.index, data['SPY']-data['SPY'][0], color='tab:blue', label='SPY Price')
ax1.plot(dates, cum_pnl, color='tab:red', label='Cumulative PnL')
ax1.set_xlabel('Date')
ax1.set_ylabel('SPY Price', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.grid(True)

# Cumulative PnL in crimson on the right axis
ax2 = ax1.twinx()
ax2.plot(data.index, data['VIX'], color='tab:green', label='VIX')
ax2.set_ylabel('VIX Index Level', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('SPY Buy & Hold PnL vs. Strategy Cumulative PnL')
plt.tight_layout()
plt.show()
